<a href="https://colab.research.google.com/github/rahul02500/Practicepython/blob/main/Assignment%20of%20Tourist%20Place%20Search%20By%20City.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain==0.3.14
!pip install langchain-openai==0.3.0
!pip install langchain-community==0.3.14

  Using cached langchain_openai-0.3.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached openai-1.109.1-py3-none-any.whl.metadata (29 kB)
Using cached langchain_openai-0.3.0-py3-none-any.whl (54 kB)
Using cached openai-1.109.1-py3-none-any.whl (948 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.8 MB/s eta 0:00:00


In [2]:
!pip install markitdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.0/70.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 64.7 MB/s eta 0:00:00


In [3]:
from google.colab import userdata

Openai_key = userdata.get('OpenAI')

In [4]:
from google.colab import userdata

Weatherapi_key = userdata.get('Weatherapi')

In [5]:
from google.colab import userdata

TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')

In [6]:
import os

os.environ['OpenAI'] = Openai_key

In [7]:
import os

os.environ['Weatherapi'] = Weatherapi_key

In [8]:
import os

os.environ['TAVILY_API_KEY'] = TAVILY_API_KEY

In [9]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_community.tools import DuckDuckGoSearchRun

In [10]:
from langchain_core.tools import tool
from markitdown import MarkItDown
from langchain_community.tools.tavily_search import TavilySearchResults
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, TimeoutError
import requests
import json
from warnings import filterwarnings
filterwarnings('ignore')

tavily_tool = TavilySearchResults(max_results=5,
                                  search_depth='advanced',
                                  include_answer=False,
                                  include_raw_content=True)
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/112.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br"
})
md = MarkItDown(requests_session=session)

@tool
def search_web_extract_info(query: str) -> list:
    """Search the web for a query and extracts useful information from the search links."""
    print('Calling web search tool')
    results = tavily_tool.invoke(query)
    docs = []

    def extract_content(url):
        """Helper function to extract content from a URL."""
        extracted_info = md.convert(url)
        text_title = extracted_info.title.strip()
        text_content = extracted_info.text_content.strip()
        return text_title + '\n' + text_content

    with ThreadPoolExecutor() as executor:
        for result in tqdm(results):
            try:
                future = executor.submit(extract_content, result['url'])
                # Wait for up to 60 seconds for the task to complete
                content = future.result(timeout=60)
                docs.append(content)
            except TimeoutError:
                print(f"Extraction timed out for url: {result['url']}")
            except Exception as e:
                print(f"Error extracting from url: {result['url']} - {e}")

    return docs


@tool
def get_weather(query: str) -> list:
    """Search weatherapi to get the current weather of the queried location."""
    print('Calling weather tool')
    base_url = "http://api.weatherapi.com/v1/current.json"
    complete_url = f"{base_url}?key={Weatherapi}&q={query}"

    response = requests.get(complete_url)
    data = response.json()
    if data.get("location"):
        return data
    else:
        return "Weather Data Not Found"

In [18]:
import os
import requests
from langchain.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_community.tools.tavily_search import TavilySearchResults


llm = ChatOpenAI(
    temperature=0,
    model="gpt-4o-mini"
)

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city"""

    api_key = os.environ["WEATHER_API_KEY"]

    url = f"http://api.weatherapi.com/v1/current.json?key={api_key}&q={city}"

    try:
        res = requests.get(url)
        data = res.json()

        return (
            f"{data['location']['name']}, {data['location']['country']} - "
            f"{data['current']['temp_c']}°C, "
            f"{data['current']['condition']['text']}"
        )

    except Exception as e:
        return f"Error: {str(e)}"

search_tool = TavilySearchResults()

@tool
def get_attractions(city: str) -> str:
    """Get top tourist attractions for a city"""

    query = f"Top tourist attractions in {city}"
    results = search_tool.run(query)

    return results

tools = [get_weather, get_attractions]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a travel assistant. Provide weather and attractions."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

if __name__ == "__main__":
    city = input("Enter city: ")

    result = agent_executor.invoke({
        "input": f"Give weather and tourist attractions in {city}"
    })

    print("\n===== OUTPUT =====")
    print(result["output"])

Enter city: Pune


> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': 'Pune'}`


Pune, India - 22.6°C, Patchy rain nearby
Invoking: `get_attractions` with `{'city': 'Pune'}`


[{'url': 'https://www.holidify.com/places/pune/sightseeing-and-things-to-do.html', 'content': '3.5  /5\n\n36 km from city center\n\nTrekking & Hiking\n\nRajgad trek is a moderate level trek that begins at the Gunjavane Village in Pune District and ends at the Rajgad Fort. It is a favourite trekking trail among intermediate level trekkers who are looking for a great location to satisfy their urge for the thrill.\n\nRead More\n\n### 22. Rajiv Gandhi Zoological Park\n\nRajiv Gandhi Zoological Park\n\n7 km from city center\n\nZoo\n\nRajiv Gandhi Zoological Park is a famous tourist attraction in Pune. The zoo expands over an area of 130 acres and is a preferred attraction for picnics and outings.\n\nRead More\n\n### 23. Bund Garden\n\nBund Garden\n\n Top Attraction   4.3  /5\n\n4 km from cit